In [35]:
# =========================
# INIT CELL (RUN FIRST ALWAYS)
# =========================

from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd


# STEP 1 — Load Cleaned Dataset

In [29]:
df = pd.read_csv("../data/filtered_complaints.csv")
print(df.shape)
print(df["Product"].value_counts())

(726799, 19)
Product
Debt collection                336076
Checking or savings account    140319
Mortgage                       130160
Credit card                     80667
Vehicle loan or lease           39577
Name: count, dtype: int64


In [5]:
import pandas as pd

product_counts = {}

for chunk in pd.read_csv(
    "../data/raw/complaints.csv",
    chunksize=100000,
    low_memory=False
):
    counts = chunk["Product"].value_counts()

    for product, count in counts.items():
        product_counts[product] = product_counts.get(product, 0) + count

print(product_counts)


{'Credit reporting or other personal consumer reports': 4834855, 'Debt collection': 799197, 'Credit card': 226686, 'Checking or savings account': 291178, 'Money transfer, virtual currency, or money service': 145066, 'Mortgage': 422254, 'Vehicle loan or lease': 72957, 'Student loan': 109717, 'Payday loan, title loan, personal loan, or advance loan': 16514, 'Prepaid card': 15280, 'Debt or credit management': 5047, 'Credit reporting, credit repair services, or other personal consumer reports': 2163857, 'Credit reporting': 140429, 'Credit card or prepaid card': 206369, 'Payday loan, title loan, or personal loan': 30641, 'Bank account or service': 86205, 'Money transfers': 5354, 'Consumer Loan': 31574, 'Payday loan': 5541, 'Other financial service': 1058, 'Virtual currency': 18}


In [7]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID', 'clean_text'],
      dtype='str')

# STEP 2 — Stratified Sampling (10K–15K)

In [8]:
sample_df = df.groupby(
    "Product",
    group_keys=False
).sample(
    frac=0.02,
    random_state=42
).reset_index(drop=True)

print(len(sample_df))

14536


In [9]:
sample_df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID', 'clean_text'],
      dtype='str')

# STEP 3 — Text Chunking

In [10]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks

In [33]:
all_chunks = []

for _, row in sample_df.iterrows():
    text = str(row["clean_text"])
    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "product": row["Product"],
            "complaint_id": row["Complaint ID"],
            "chunk_index": i
        })

chunks_df = pd.DataFrame(all_chunks)
print(chunks_df.shape)

(42867, 4)


# STEP 4 — Load Embedding Model

In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1828.84it/s]


Model loaded


# STEP 5 — Generate Embeddings

In [19]:
embeddings = model.encode(
    chunks_df["text"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 1340/1340 [44:42<00:00,  2.00s/it] 


In [20]:
print(embeddings.shape)

(42867, 384)


# STEP 6 — Create Vector Store (FAISS method — recommended)

In [21]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# STEP 7 — Save Vector Store

In [22]:
import os

os.makedirs("vector_store", exist_ok=True)

faiss.write_index(
    index,
    "vector_store/faiss_index.bin"
)

chunks_df.to_csv(
    "vector_store/chunk_metadata.csv",
    index=False
)

# STEP 8- Test retrieval

In [24]:
def retrieve(query, k=5):
    query_vec = model.encode([query])
    D, I = index.search(query_vec, k)
    return chunks_df.iloc[I[0]]

In [25]:
print(type(model))
print(type(index))
print(type(chunks_df))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
<class 'faiss.swigfaiss.IndexFlatL2'>
<class 'pandas.DataFrame'>


In [34]:
retrieve("credit card billing problem")

,text,product,complaint_id,chunk_index
11667,i have had this credit card for many years and...,Credit card,12803029,0
11228,my only american express account number is xxx...,Credit card,1548607,0
8672,everytime i try to pay the credit card on time...,Credit card,12131829,0
11490,i missed one payment on my american express ca...,Credit card,2103335,0
9723,xxxx credit card opened account on xxxxyear wa...,Credit card,9396602,0


## Task 2: Text Chunking, Embedding, and Vector Store Indexing

### Sampling Strategy

A stratified sampling approach was used to preserve the proportional distribution of complaint records across the five product categories: Credit Card, Mortgage, Debt Collection, Checking or Savings Account, and Vehicle Loan or Lease. A sampling fraction of 0.02 was applied to each category, resulting in a final sample of 14,536 complaints. This approach ensured that all product categories were represented while reducing computational cost during embedding generation.

### Chunking Strategy

Customer complaint narratives were divided into smaller chunks using a custom chunking function. A chunk size of 500 characters and an overlap of 50 characters were selected. The overlap helps preserve context between neighboring chunks and reduces information loss at chunk boundaries. This improves the quality of semantic retrieval during the RAG process.

### Embedding Model Choice

The `sentence-transformers/all-MiniLM-L6-v2` model was selected to generate embeddings. This model produces 384-dimensional vectors, offers strong semantic similarity performance, and is lightweight enough to run efficiently on standard hardware. It is widely used as a baseline model for Retrieval-Augmented Generation systems.

### Vector Store Creation

Embeddings were indexed using FAISS with the `IndexFlatL2` similarity metric. The vector store enables efficient semantic search over complaint narratives. Metadata including complaint ID, product category, and chunk index was stored separately in `chunk_metadata.csv`, allowing retrieved chunks to be traced back to their original complaints.
